In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

processed_dir = Path("processed_data")

gases = ["ethanol", "ammonia", "acetone", "toluol"]

all_features = []

for gas in gases:

    print(f"\nProcessing: {gas}")

    sensor_file = processed_dir / f"{gas}.csv"
    sensor_df = pd.read_csv(sensor_file)

    concentration_col = sensor_df.columns[0]
    ug_columns = sensor_df.columns[1:]

    concentrations = pd.to_numeric(
        sensor_df.iloc[:, 0],
        errors='coerce'
    ).values

    add_file = processed_dir / f"{gas}_add.csv"

    add_df = pd.read_csv(add_file)

    add_df.columns = ["concentration", "rs_rs0"]

    add_df["concentration"] = pd.to_numeric(
        add_df["concentration"],
        errors='coerce'
    )

    add_df["rs_rs0"] = pd.to_numeric(
        add_df["rs_rs0"],
        errors='coerce'
    )

    for i, conc in enumerate(concentrations):

        if np.isnan(conc):
            continue

        feature_vector = pd.to_numeric(
            sensor_df.iloc[i, 1:],
            errors='coerce'
        ).values.tolist()

        rs_match = add_df[
            np.isclose(add_df["concentration"], conc)
        ]

        if rs_match.empty:
            print(f"Rs/Rs0 not found: {gas}, C={conc}")
            continue

        rs_value = rs_match.iloc[0]["rs_rs0"]

        feature_vector.append(rs_value)

        row = {
            "gas": gas,
            "C, % \\ Ug, V": conc
        }

        for j, val in enumerate(feature_vector[:-1]):
            ug_name = ug_columns[j]
            row[str(ug_name)] = val

        row["Rs/Rs0"] = rs_value

        all_features.append(row)

features_df = pd.DataFrame(all_features)

features_df.to_csv("feature_vectors.csv", index=False)

print("\nDONE!")
print("Saved: feature_vectors.csv")


Processing: ethanol

Processing: ammonia

Processing: acetone

Processing: toluol

DONE!
Saved: feature_vectors.csv
